In [2]:
import pandas as pd
import numpy as np

import vitaldb as v

# Preprocessing

In [1]:
track_names = ["SNUADC/ART",        # Arterial pressure wave  | W/500 | mmHg
               "SNUADC/ECG_II",     # ECG lead II wave        | W/500 | mV
               "SNUADC/ECG_V5",     # ECG lead V5 wave        | W/500 | mV
               "BIS/EEG1_WAV",      # EEG wave from channel 1 | W/128 | uV
               "BIS/EEG2_WAV",      # EEG wave from channel 2 | W/128 | uV
               "Solar8000/RR_CO2",  # Respiratory rate based on capnography | N | /min
               "Primus/CO2",        # Capnography wave        | W/62.5 | mmHg
               "BIS/BIS",           # Bispectral index value  |    N   | unitless
               ]

# Dataset functions

In [14]:
def load_normalized_eeg_samples(path, seq_len, features):
    df = pd.read_csv(path)
    x = []
    x_means = {}
    x_stds = {}
    for f in features:
        x_f = df[f]
        x_f_mean = np.mean(x_f)
        x_f_std = np.std(x_f)
        x_f_z = [(x_fi - x_f_mean)/x_f_std for x_fi in x_f]
        x.append(x_f_z)
        x_means[f] = x_f_mean
        x_stds[f] = x_f_std
    
    x_z = np.stack(x, axis=1)
    X = np.array([x_z[i:i+seq_len] for i in range(x_z.shape[0]-seq_len)])
    X = X.reshape(-1, seq_len, len(features))

    y = df.bis
    Y = np.array([y[i+seq_len] for i in range(len(y)-seq_len)]).reshape(-1, 1)

    return X, Y, x_means, x_stds

In [15]:
def load_normalized_eeg_train(path, split, seq_len, features):
    X, Y, x_means, x_stds = load_normalized_eeg_samples(path, seq_len, features)
    split_idx = int(len(X) * split)
    return X[:split_idx], Y[:split_idx], x_means, x_stds

In [16]:
path = '../data/processed/eeg_sample.csv'
split = 0.8
seq_len = 200
features = ['delta','theta','alpha','beta']
X_train, Y_train, x_means, x_stds = load_normalized_eeg_train(path, split, seq_len, features)